In [5]:
# 1. ALL IMPORTS AT THE TOP
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import time
from warnings import filterwarnings

filterwarnings('ignore')

In [6]:
# 2. DEFINE LENET-5 (Required for Task 2a)
class LeNet5(nn.Module):
    def __init__(self, in_channels=1):
        super(LeNet5, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 6, kernel_size=5), nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(6, 16, kernel_size=5), nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(16 * 5 * 5, 120), nn.ReLU(),
            nn.Linear(120, 84), nn.ReLU(),
            nn.Linear(84, 10)
        )
    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

In [7]:
# 3. HELPER FUNCTION TO GET DATA (Task 1)
def get_data(dataset_name):
    # Normalize pixel values to [0,1] as per Task 1
    if dataset_name == "MNIST":
        # Resize to 32x32 so it matches CIFAR-10 dimensions for the models
        transform = transforms.Compose([transforms.Resize((32,32)), transforms.ToTensor()])
        trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
        testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
        in_channels = 1
    else: # CIFAR-10
        transform = transforms.Compose([transforms.ToTensor()])
        trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
        testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
        in_channels = 3

    return trainset, testset, in_channels

# 4. MAIN EXECUTION (Tasks 2, 3, and 4)
results = []
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for d_name in ["MNIST", "CIFAR-10"]:
    trainset, testset, in_ch = get_data(d_name)
    trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)

    for m_name in ["LeNet", "AlexNet", "ResNet"]:
        print(f"Running {m_name} on {d_name}...")

        # Build Model (Task 2)
        if m_name == "LeNet":
            model = LeNet5(in_channels=in_ch)
        elif m_name == "AlexNet":
            model = models.alexnet(num_classes=10)
            model.features[0] = nn.Conv2d(in_ch, 64, kernel_size=3, stride=1, padding=1)
        elif m_name == "ResNet":
            model = models.resnet18(num_classes=10)
            model.conv1 = nn.Conv2d(in_ch, 64, kernel_size=3, stride=1, padding=1, bias=False)

        model = model.to(device)

        # Stats (Task 4)
        num_params = sum(p.numel() for p in model.parameters())
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001)

        # Training Loop (Task 3)
        start_time = time.time()
        model.train()
        for epoch in range(5): # Set to 5 or 10 epochs
            for imgs, labels in trainloader:
                imgs, labels = imgs.to(device), labels.to(device)
                optimizer.zero_grad()
                loss = criterion(model(imgs), labels)
                loss.backward()
                optimizer.step()

        end_time = time.time()

        results.append({
            "Dataset": d_name,
            "Model": m_name,
            "Parameters": num_params,
            "Time": round(end_time - start_time, 2)
        })

Running LeNet on MNIST...
Running AlexNet on MNIST...
Running ResNet on MNIST...
Running LeNet on CIFAR-10...
Running AlexNet on CIFAR-10...
Running ResNet on CIFAR-10...


In [8]:
# 5. TABULATE RESULTS (Task 4 output)
print("\n--- FINAL RESULTS ---")
print(f"{'Dataset':<10} | {'Model':<10} | {'Params':<12} | {'Time (s)':<10}")
for r in results:
    print(f"{r['Dataset']:<10} | {r['Model']:<10} | {r['Parameters']:<12,} | {r['Time']:<10}")


--- FINAL RESULTS ---
Dataset    | Model      | Params       | Time (s)  
MNIST      | LeNet      | 61,706       | 46.43     
MNIST      | AlexNet    | 57,022,154   | 129.21    
MNIST      | ResNet     | 11,172,810   | 101.59    
CIFAR-10   | LeNet      | 62,006       | 34.48     
CIFAR-10   | AlexNet    | 57,023,306   | 108.04    
CIFAR-10   | ResNet     | 11,173,962   | 85.06     
